In [1]:
import cv2
import numpy as np
import joblib
from skimage import feature

In [2]:
# =============================================================================
# 1. CONFIGURACIÓN Y CARGA DE MODELOS (Idéntico a tu notebook)
# =============================================================================

# Rutas a los modelos entrenados en la Tarea I
SVM_PATH = "models/svm_lbp_barba.joblib"
SCALER_PATH = "models/scaler_lbp_barba.joblib"

# Cargar modelos
try:
    svm_lbp = joblib.load(SVM_PATH)
    scaler_lbp = joblib.load(SCALER_PATH)
    print("Modelos cargados correctamente.")
except FileNotFoundError:
    print(f"Error: No se encuentran los modelos en {SVM_PATH} o {SCALER_PATH}")
    exit()

# Parámetros fijos del entrenamiento (NO CAMBIAR o el SVM fallará)
WIDTH = 192
HEIGHT = 192
CLASS_LABELS = ["con_barba", "sin_barba"] # [0, 1]

# Detector de caras de OpenCV
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


Modelos cargados correctamente.


In [3]:
# =============================================================================
# 2. FUNCIONES DE SOPORTE (Reutilizadas del Notebook)
# =============================================================================

def lbphist(gray, ncellsx, ncellsy, width, height, lbp_method):
    """Calcula el histograma LBP exactamente como en el entrenamiento."""
    pxpercellx = int(width / ncellsx)
    pxpercelly = int(height / ncellsy)
    ofx = int((width - int(pxpercellx) * ncellsx) / 2)
    ofy = int((height - int(pxpercelly) * ncellsy) / 2)
    LBPu_hist = []
    for i in range(ncellsy):
        for j in range(ncellsx):
            roi = gray[ofy + i * pxpercelly:ofy + (i + 1) * pxpercelly,
                       ofx + j * pxpercellx:ofx + (j + 1) * pxpercellx]
            lbpimg = feature.local_binary_pattern(roi, 8, 1, method=lbp_method)
            n_bins = 256
            feath, _ = np.histogram(lbpimg, density=False,
                                    bins=n_bins, range=(0, n_bins))
            LBPu_hist = np.concatenate([LBPu_hist, feath])
    return LBPu_hist

def preprocess_lbp_from_gray(gray):
    """Redimensiona, extrae LBP y escala los datos."""
    gray_resized = cv2.resize(gray, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)
    feat_lbp = lbphist(gray_resized,
                       ncellsx=3, ncellsy=3,
                       width=WIDTH, height=HEIGHT,
                       lbp_method="nri_uniform")
    desc = feat_lbp.astype("float32").reshape(1, -1)
    desc_scaled = scaler_lbp.transform(desc)
    return desc_scaled

def expand_bbox(x, y, w, h, frame_width, frame_height, factor=1.3):
    """Expande el bounding box para dar contexto (cuello/pelo) al clasificador."""
    cx = x + w / 2.0
    cy = y + h / 2.0
    new_w = w * factor
    new_h = h * factor
    x_new = max(0, int(cx - new_w / 2.0))
    y_new = max(0, int(cy - new_h / 2.0))
    x_new2 = min(frame_width,  int(x_new + new_w))
    y_new2 = min(frame_height, int(y_new + new_h))
    return x_new, y_new, x_new2 - x_new, y_new2 - y_new


In [4]:
# =============================================================================
# 3. FUNCIONES PARA EL FILTRO (OVERLAY)
# =============================================================================

def overlay_image_alpha(img, img_overlay, x, y, w_overlay, h_overlay):
    """
    Superpone una imagen PNG (img_overlay) sobre el frame (img) respetando el canal Alpha.
    x, y: coordenadas top-left donde poner el overlay.
    w_overlay, h_overlay: tamaño al que redimensionar el overlay.
    """
    # Copiamos para no modificar el original si no queremos
    # img = img.copy() 
    
    # Dimensiones del fondo
    h, w, _ = img.shape

    # Redimensionar la imagen superpuesta
    img_overlay = cv2.resize(img_overlay, (w_overlay, h_overlay))

    # Recortes para manejar bordes de pantalla (clipping)
    y1, y2 = max(0, y), min(h, y + h_overlay)
    x1, x2 = max(0, x), min(w, x + w_overlay)

    # Si el overlay se sale completamente, no hacer nada
    if y1 >= y2 or x1 >= x2:
        return img

    # Calcular offsets para la imagen superpuesta (si se corta por arriba o izq)
    y_overlay_start = max(0, -y)
    x_overlay_start = max(0, -x)
    y_overlay_end = y_overlay_start + (y2 - y1)
    x_overlay_end = x_overlay_start + (x2 - x1)

    overlay_crop = img_overlay[y_overlay_start:y_overlay_end, x_overlay_start:x_overlay_end]
    background_crop = img[y1:y2, x1:x2]

    # Separar canales: Color y Alpha
    alpha_mask = overlay_crop[:, :, 3] / 255.0
    alpha_inv = 1.0 - alpha_mask

    # Mezclar
    for c in range(0, 3):
        background_crop[:, :, c] = (alpha_mask * overlay_crop[:, :, c] +
                                    alpha_inv * background_crop[:, :, c])

    # Poner el parche mezclado en la imagen original
    img[y1:y2, x1:x2] = background_crop
    return img



In [6]:
# =============================================================================
# 4. BUCLE PRINCIPAL
# =============================================================================

def main():
    # Cargar recursos gráficos (asegúrate de tener estas imágenes)
    try:
        # Lee las imágenes conservando el canal Alpha (-1 o cv2.IMREAD_UNCHANGED)
        img_saiyan = cv2.imread("filter_assets/super_saiyan_hair.png", -1)
        img_kawaii_hair = cv2.imread("filter_assets/anime_girl_hair.png", -1)
        img_kawaii_eyes = cv2.imread("filter_assets/anime_girl_eyes.png", -1)
        
        if img_saiyan is None or img_kawaii_hair is None:
            raise FileNotFoundError("No se pudieron cargar las imágenes PNG.")
            
    except Exception as e:
        print(f"Error cargando assets: {e}")
        print("Asegúrate de tener 'saiyan_hair.png' y 'kawaii_hair.png' en la carpeta.")
        return

    cap = cv2.VideoCapture(0)

    print("Iniciando filtro. Pulsa 'ESC' para salir.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h_frame, w_frame = frame_gray.shape

        # Detectar caras (Viola-Jones standard)
        faces = face_cascade.detectMultiScale(frame_gray, scaleFactor=1.3, minNeighbors=5)

        for (x, y, w, h) in faces:
            # 1. Clasificación (Barba vs No Barba)
            # Expandimos bbox para inferencia (igual que en el notebook)
            x_exp, y_exp, w_exp, h_exp = expand_bbox(x, y, w, h, w_frame, h_frame, factor=1.3)
            face_roi = frame_gray[y_exp:y_exp+h_exp, x_exp:x_exp+w_exp]
            
            # Predecir
            try:
                desc = preprocess_lbp_from_gray(face_roi)
                pred_idx = int(svm_lbp.predict(desc)[0])
                label = CLASS_LABELS[pred_idx]
            except Exception:
                # Si el recorte se sale o falla, asumimos una clase por defecto
                label = "sin_barba"

            # 2. Aplicación del Filtro
            if label == "con_barba":
                # --- MODO SUPER SAIYAN ---
                # Ajuste de posición del pelo (encima de la cabeza)
                hair_w = int(w * 1.8)  # El pelo saiyan es muy voluminoso
                hair_h = int(h * 1.5)
                
                # Centrado respecto a la cara
                hair_x = x + w // 2 - hair_w // 2
                # Offset vertical (un poco hacia arriba de la frente)
                hair_y = y - int(h * 0.9) 

                overlay_image_alpha(frame, img_saiyan, hair_x, hair_y, hair_w, hair_h)
                
                # Texto informativo
                cv2.putText(frame, "SAIYAN MODE", (x, y + h + 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            else:
                # --- MODO KAWAII ---
                # 1. Pelo Kawaii
                hair_w = int(w * 1.4)
                hair_h = int(h * 1.2)
                hair_x = x + w // 2 - hair_w // 2
                hair_y = y - int(h * 0.45) # Ajuste para que encaje en la frente

                overlay_image_alpha(frame, img_kawaii_hair, hair_x, hair_y, hair_w, hair_h)

                # 2. Ojos Kawaii (Estimación simple sin landmarks complejos)
                # Asumimos que los ojos están más o menos al 35% de la altura de la cara
                if img_kawaii_eyes is not None:
                    eyes_w = int(w * 0.9)
                    eyes_h = int(h * 0.4)
                    eyes_x = x + w // 2 - eyes_w // 2
                    eyes_y = y + int(h * 0.25) 
                    
                    overlay_image_alpha(frame, img_kawaii_eyes, eyes_x, eyes_y, eyes_w, eyes_h)

                cv2.putText(frame, "KAWAII MODE", (x, y + h + 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (203, 192, 255), 2)

        cv2.imshow("Filtro VC P5 - Barba Detector", frame)

        if cv2.waitKey(1) & 0xFF == 27: # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Iniciando filtro. Pulsa 'ESC' para salir.
